# 07 · Forecasting From a CSV File

Real data usually lives in a CSV. Here we load one with pandas, forecast the
numeric columns, and also show the ready-made CLI shipped in the repo.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

## Create a demo CSV (replace with your own file)

In [ ]:
import pandas as pd
rng = np.random.default_rng(11)
dates = pd.date_range("2022-01-01", periods=180, freq="D")
df = pd.DataFrame({
    "date": dates,
    "sales":   np.clip(300 + 3*np.arange(180) + 40*np.sin(2*np.pi*np.arange(180)/7) + rng.normal(0,15,180), 0, None),
    "traffic": np.clip(900 + 60*np.sin(2*np.pi*np.arange(180)/7) + rng.normal(0,40,180), 0, None),
})
df.to_csv("demo_sales.csv", index=False)
df.head()

## Load & forecast each numeric column

In [ ]:
value_cols = ["sales", "traffic"]
inputs = [df[c].dropna().values.astype(np.float32) for c in value_cols]

horizon = 21
point, q = model.forecast(horizon=horizon, inputs=inputs)

# Build a tidy future dataframe with dates
future_dates = pd.date_range(df["date"].iloc[-1], periods=horizon+1, freq="D")[1:]
out = []
for i, col in enumerate(value_cols):
    for h in range(horizon):
        out.append({
            "date": future_dates[h], "series": col,
            "forecast": round(float(point[i, h]), 2),
            "lower_80": round(float(q[i, h, 1]), 2),
            "upper_80": round(float(q[i, h, 9]), 2),
        })
forecast_df = pd.DataFrame(out)
forecast_df.to_csv("demo_sales_forecast.csv", index=False)
forecast_df.head(10)

## Or use the built-in CLI (no code)

The repo ships an end-to-end CSV forecaster:

```bash
python ../timesfm-forecasting/scripts/forecast_csv.py demo_sales.csv \
    --horizon 21 \
    --date-col date \
    --value-cols sales,traffic \
    --output demo_forecast.csv
```

In [ ]:
!python ../timesfm-forecasting/scripts/forecast_csv.py demo_sales.csv \
    --horizon 21 --date-col date --value-cols sales,traffic \
    --output demo_forecast_cli.csv --skip-check